# Normalizaci?n y preparaci?n de cotizaciones

## Objetivo

Limpiar cotizaciones de proveedores en Excel, clasificar sus filas, asignar apartados, agrupar partidas duplicadas y exportar una tabla de cuatro columnas para Airtable.

## Principios

- **Trazabilidad:** conservar `Original_Order` para relacionar cada fila con su posici?n en el archivo fuente.
- **Nada se borra sin rastro:** `df_original` preserva la entrada y las filas excluidas se contabilizan por su orden original.
- **Integridad:** las comprobaciones del pipeline se expresan como `assert` con mensajes claros.

## Configuraci?n

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

CURRENT_DIR = Path.cwd()
REPO_ROOT = next(
    (
        path
        for path in (CURRENT_DIR, *CURRENT_DIR.parents)
        if (path / "data").is_dir() and (path / "notebooks").is_dir()
    ),
    CURRENT_DIR,
)
INPUT_PATH = REPO_ROOT / "data" / "sample_input.xlsx"
OUTPUT_PATH = REPO_ROOT / "output" / "sample_output.xlsx"
SHEET = 0
HEADER_ROW = 1

COLUMNS = {
    "unnamed": "Unnamed: 0",
    "csi": "CSI",
    "labor_code": "Labor Code",
    "description": "Scope Description",
    "bid_qty": "Bid Form QTY",
    "bid_uom": "Bid Form UOM",
    "qty": "QTY",
    "uom": "UOM",
    "unit_rate": "Unit Rate",
    "total": "Total",
    "comments": "Comments/ Notes",
    "gs": "GS",
    "original_order": "Original_Order",
    "row_type": "Row_Type",
    "section": "Section",
    "final_description": "Scope Description Final",
}
HEADER_COLUMNS = [
    COLUMNS[name]
    for name in (
        "unnamed",
        "csi",
        "labor_code",
        "description",
        "bid_qty",
        "bid_uom",
        "qty",
        "uom",
        "unit_rate",
        "total",
        "comments",
        "gs",
    )
]
TEXT_COLUMNS = [COLUMNS["description"], COLUMNS["uom"]]
SUMMARY_ROWS = ["Sub Total", "Package Total", "Mark Ups"]
EMPTY_RATE_MARKER = "$-"
ROW_TYPES = {
    "complete": "Detail_Complete",
    "missing_rate": "Detail_Missing_Rate",
    "section": "Section_Candidate",
    "blank": "Blank_Row",
    "partial": "Partial_Record",
}
DIAGNOSTICO = False

## Paso 1. Leer y preservar el archivo original

In [2]:
df_original = pd.read_excel(
    INPUT_PATH,
    sheet_name=SHEET,
    header=HEADER_ROW,
)
assert df_original.columns.tolist() == HEADER_COLUMNS, (
    "Los encabezados de entrada no coinciden con la estructura esperada."
)
df_original[COLUMNS["original_order"]] = np.arange(len(df_original))
df = df_original.copy()

assert df_original[COLUMNS["original_order"]].is_unique, (
    "Original_Order debe identificar de forma ?nica cada fila fuente."
)

## Paso 2. Normalizar texto y retirar vac?os y res?menes

In [3]:
for column in TEXT_COLUMNS:
    df[column] = df[column].astype("string").str.strip()

fields_for_empty_check = [
    COLUMNS["description"],
    COLUMNS["qty"],
    COLUMNS["unit_rate"],
    COLUMNS["uom"],
]
empty_mask = df[fields_for_empty_check].isna().all(axis=1)
empty_orders = df.loc[empty_mask, COLUMNS["original_order"]].tolist()
df = df.loc[~empty_mask].copy()

summary_mask = df[COLUMNS["description"]].isin(SUMMARY_ROWS)
summary_orders = df.loc[summary_mask, COLUMNS["original_order"]].tolist()
df = df.loc[~summary_mask].copy().reset_index(drop=True)

removed_orders = sorted(empty_orders + summary_orders)
retained_orders = sorted(df[COLUMNS["original_order"]].tolist())
assert sorted(removed_orders + retained_orders) == list(
    range(len(df_original))
), "Las filas descartadas y conservadas deben cubrir la fuente sin omisiones."
assert df[COLUMNS["original_order"]].is_unique, (
    "Original_Order debe seguir siendo ?nico tras la limpieza."
)
assert df[COLUMNS["original_order"]].is_monotonic_increasing, (
    "La limpieza debe conservar el orden de las filas fuente."
)
assert not df[COLUMNS["description"]].isin(SUMMARY_ROWS).any(), (
    "No deben quedar filas de resumen en el conjunto de trabajo."
)
print(f"Filas vac?as identificadas: {len(empty_orders)}")
print(f"Filas de resumen identificadas: {len(summary_orders)}")

Filas vac?as identificadas: 1
Filas de resumen identificadas: 3


## Paso 3. Normalizar el precio vac?o y clasificar las filas

In [4]:
rate_column = COLUMNS["unit_rate"]
empty_rate_mask = (
    df[rate_column]
    .astype("string")
    .str.strip()
    .eq(EMPTY_RATE_MARKER)
)
df.loc[empty_rate_mask, rate_column] = np.nan
print(f'Marcadores "{EMPTY_RATE_MARKER}" normalizados: {int(empty_rate_mask.sum())}')

description_present = df[COLUMNS["description"]].notna()
qty_present = df[COLUMNS["qty"]].notna()
rate_present = df[COLUMNS["unit_rate"]].notna()
uom_present = df[COLUMNS["uom"]].notna()

df[COLUMNS["row_type"]] = np.select(
    [
        description_present & qty_present & rate_present & uom_present,
        description_present & qty_present & ~rate_present & uom_present,
        description_present & ~qty_present & ~rate_present & ~uom_present,
        ~description_present & ~qty_present & ~rate_present & ~uom_present,
    ],
    [
        ROW_TYPES["complete"],
        ROW_TYPES["missing_rate"],
        ROW_TYPES["section"],
        ROW_TYPES["blank"],
    ],
    default=ROW_TYPES["partial"],
)

display(
    df[COLUMNS["row_type"]]
    .value_counts()
    .rename_axis(COLUMNS["row_type"])
    .reset_index(name="Filas")
)

Marcadores "$-" normalizados: 1


,Row_Type,Filas
0,Detail_Complete,15
1,Section_Candidate,3
2,Detail_Missing_Rate,1


## Paso 4. Asignar el apartado a cada partida

In [5]:
section_candidate = df[COLUMNS["row_type"]].eq(ROW_TYPES["section"])
df[COLUMNS["section"]] = np.where(
    section_candidate,
    df[COLUMNS["description"]],
    np.nan,
)
df[COLUMNS["section"]] = df[COLUMNS["section"]].ffill()
df.loc[section_candidate, COLUMNS["section"]] = np.nan

assert df.loc[section_candidate, COLUMNS["section"]].isna().all(), (
    "Las filas de encabezado no deben asignarse a s? mismas como apartado."
)

## Paso 5. Diferenciar descripciones repetidas entre apartados

In [6]:
record_types = [
    ROW_TYPES["complete"],
    ROW_TYPES["missing_rate"],
    ROW_TYPES["partial"],
]
records = df.loc[
    df[COLUMNS["row_type"]].isin(record_types)
    & df[COLUMNS["description"]].notna()
].copy()
sections_per_description = records.groupby(COLUMNS["description"])[
    COLUMNS["section"]
].nunique()
ambiguous_descriptions = sections_per_description[
    sections_per_description.gt(1)
].index
ambiguous_mask = (
    df[COLUMNS["row_type"]].isin(record_types)
    & df[COLUMNS["description"]].isin(ambiguous_descriptions)
    & df[COLUMNS["section"]].notna()
)

df[COLUMNS["final_description"]] = df[COLUMNS["description"]]
df.loc[ambiguous_mask, COLUMNS["final_description"]] = (
    df.loc[ambiguous_mask, COLUMNS["description"]]
    + " / "
    + df.loc[ambiguous_mask, COLUMNS["section"]]
)

assert df.loc[~ambiguous_mask, COLUMNS["final_description"]].equals(
    df.loc[~ambiguous_mask, COLUMNS["description"]]
), "Las descripciones no ambiguas deben conservarse sin cambios."
if ambiguous_mask.any():
    assert df.loc[ambiguous_mask, COLUMNS["final_description"]].notna().all(), (
        "Toda descripci?n ambigua debe incluir su apartado en el texto final."
    )
display(
    df.loc[
        ambiguous_mask,
        [
            COLUMNS["original_order"],
            COLUMNS["description"],
            COLUMNS["section"],
            COLUMNS["final_description"],
        ],
    ]
)

,Original_Order,Scope Description,Section,Scope Description Final
3,3,Temporary fencing,General Conditions,Temporary fencing / General Conditions
4,4,Temporary fencing,General Conditions,Temporary fencing / General Conditions
17,17,Temporary fencing,Electrical,Temporary fencing / Electrical


## Paso 6. Agrupar partidas completas duplicadas

In [7]:
detail_complete = df.loc[
    df[COLUMNS["row_type"]].eq(ROW_TYPES["complete"])
].copy()
grouping_key = [
    COLUMNS["final_description"],
    COLUMNS["unit_rate"],
    COLUMNS["uom"],
]
group_sizes = detail_complete.groupby(
    grouping_key,
    dropna=False,
).size()
merged_rows = int((group_sizes - 1).sum())

detail_grouped = (
    detail_complete.groupby(
        grouping_key,
        as_index=False,
        sort=False,
        dropna=False,
    )
    .agg(
        **{
            COLUMNS["original_order"]: (COLUMNS["original_order"], "min"),
            COLUMNS["qty"]: (COLUMNS["qty"], "sum"),
        }
    )
    .rename(columns={COLUMNS["final_description"]: COLUMNS["description"]})
)
rows_not_grouped = df.loc[
    ~df[COLUMNS["row_type"]].eq(ROW_TYPES["complete"])
].copy()
rows_not_grouped[COLUMNS["description"]] = rows_not_grouped[
    COLUMNS["final_description"]
]
rows_not_grouped = rows_not_grouped[
    [
        COLUMNS["original_order"],
        COLUMNS["description"],
        COLUMNS["qty"],
        COLUMNS["unit_rate"],
        COLUMNS["uom"],
    ]
]

df_export_auditoria = (
    pd.concat([detail_grouped, rows_not_grouped], ignore_index=True)
    .sort_values(COLUMNS["original_order"], kind="stable")
    .reset_index(drop=True)
)
df_final = df_export_auditoria[
    [
        COLUMNS["description"],
        COLUMNS["qty"],
        COLUMNS["unit_rate"],
        COLUMNS["uom"],
    ]
].copy()

expected_final_rows = len(df) - merged_rows
assert len(df_final) == expected_final_rows, (
    "El n?mero de filas finales debe ser el original limpio menos las filas fusionadas."
)
assert df_export_auditoria[COLUMNS["original_order"]].is_monotonic_increasing, (
    "La tabla de auditor?a debe conservar el orden fuente de cada grupo."
)
assert df_final.columns.tolist() == [
    COLUMNS["description"],
    COLUMNS["qty"],
    COLUMNS["unit_rate"],
    COLUMNS["uom"],
], "La exportaci?n debe contener ?nicamente las cuatro columnas requeridas."
print(f"Filas completas agrupadas: {len(detail_complete)}")
print(f"Filas fusionadas: {merged_rows}")
print(f"Filas de salida: {len(df_final)}")
display(df_final.head(30))

Filas completas agrupadas: 15
Filas fusionadas: 2
Filas de salida: 17


,Scope Description,QTY,Unit Rate,UOM
0,General Conditions,NaN,NaN,<NA>
1,Mobilization,1,5000,LS
2,Site protection,1,1200,LS
3,Temporary fencing / General Conditions,150,15,LF
4,Existing conditions survey,1,700,LS
5,Mechanical,NaN,NaN,<NA>
6,Demolish existing piping,200,8,LF
7,Install copper piping,140,25,LF
8,Pipe insulation,140,NaN,LF
9,System startup,1,1500,LS


## Paso 7. Validar totales de control

In [8]:
original_qty_numeric = pd.to_numeric(
    detail_complete[COLUMNS["qty"]],
    errors="coerce",
)
original_rate_numeric = pd.to_numeric(
    detail_complete[COLUMNS["unit_rate"]],
    errors="coerce",
)
original_numeric_mask = original_qty_numeric.notna() & original_rate_numeric.notna()

aggregated_qty_numeric = pd.to_numeric(
    detail_grouped[COLUMNS["qty"]],
    errors="coerce",
)
aggregated_rate_numeric = pd.to_numeric(
    detail_grouped[COLUMNS["unit_rate"]],
    errors="coerce",
)
aggregated_numeric_mask = (
    aggregated_qty_numeric.notna() & aggregated_rate_numeric.notna()
)

original_qty_total = original_qty_numeric[original_numeric_mask].sum()
aggregated_qty_total = aggregated_qty_numeric[aggregated_numeric_mask].sum()
original_amount_total = (
    original_qty_numeric[original_numeric_mask]
    * original_rate_numeric[original_numeric_mask]
).sum()
aggregated_amount_total = (
    aggregated_qty_numeric[aggregated_numeric_mask]
    * aggregated_rate_numeric[aggregated_numeric_mask]
).sum()

assert np.isclose(
    original_qty_total,
    aggregated_qty_total,
    rtol=1e-9,
    atol=1e-9,
), "La suma de QTY debe conservarse tras agrupar las partidas completas num?ricas."
assert np.isclose(
    original_amount_total,
    aggregated_amount_total,
    rtol=1e-9,
    atol=1e-9,
), "La suma de QTY x Unit Rate debe conservarse tras agrupar."

original_non_numeric_qty = int(original_qty_numeric.isna().sum())
aggregated_non_numeric_qty = int(aggregated_qty_numeric.isna().sum())
assert original_non_numeric_qty == aggregated_non_numeric_qty, (
    "Las cantidades no num?ricas deben permanecer representadas tras agrupar."
)
print(f"Total de QTY num?rico controlado: {original_qty_total}")
print(f"Total de QTY x Unit Rate controlado: {original_amount_total}")
print(f"Cantidades completas no num?ricas preservadas: {original_non_numeric_qty}")

Total de QTY num?rico controlado: 652.0
Total de QTY x Unit Rate controlado: 20465.0
Cantidades completas no num?ricas preservadas: 1


## Paso 8. Exportar y verificar el archivo para Airtable

In [9]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_final.to_excel(
    OUTPUT_PATH,
    index=False,
    engine="openpyxl",
)
verified_output = pd.read_excel(OUTPUT_PATH)
assert verified_output.columns.tolist() == df_final.columns.tolist(), (
    "El archivo exportado debe conservar las cuatro columnas de Airtable."
)
assert len(verified_output) == len(df_final), (
    "El archivo exportado debe conservar el n?mero de filas de la tabla final."
)
print(f"Archivo creado: {OUTPUT_PATH.relative_to(REPO_ROOT)}")
print(f"Filas exportadas: {len(verified_output)}")
display(verified_output.head(30))

Archivo creado: output\sample_output.xlsx
Filas exportadas: 17


,Scope Description,QTY,Unit Rate,UOM
0,General Conditions,NaN,NaN,NaN
1,Mobilization,1,5000.0,LS
2,Site protection,1,1200.0,LS
3,Temporary fencing / General Conditions,150,15.0,LF
4,Existing conditions survey,1,700.0,LS
5,Mechanical,NaN,NaN,NaN
6,Demolish existing piping,200,8.0,LF
7,Install copper piping,140,25.0,LF
8,Pipe insulation,140,NaN,LF
9,System startup,1,1500.0,LS


## Ap?ndice. Diagn?stico opcional

In [10]:
if DIAGNOSTICO:
    display(
        df.loc[
            df[COLUMNS["qty"]].isna(),
            [COLUMNS["description"], COLUMNS["qty"], COLUMNS["unit_rate"], COLUMNS["uom"]],
        ]
    )
    display(
        df.loc[
            df[COLUMNS["unit_rate"]].isna(),
            [COLUMNS["description"], COLUMNS["qty"], COLUMNS["unit_rate"], COLUMNS["uom"]],
        ]
    )
    display(
        df.loc[
            df[COLUMNS["uom"]].isna(),
            [COLUMNS["description"], COLUMNS["qty"], COLUMNS["unit_rate"], COLUMNS["uom"]],
        ]
    )